# Predicting Student Health Risk — LightGBM + Optuna Tuning

Started with a RandomForest baseline just to get something on the board (scored ~0.86 balanced accuracy locally, was way behind everyone else's LightGBM/CatBoost scores). Switched to LightGBM and jumped straight to ~0.949 CV, then ran Optuna for a bit to squeeze out a bit more.

This version scored 0.95011 on the public leaderboard, which is my best so far after also trying an ensemble with CatBoost, some feature engineering, and pseudo-labeling — none of which actually beat this.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import optuna
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = "/kaggle/input/competitions/playground-series-s6e7"


## 1. Load and preprocess

Loading the raw data and splitting features from the target.

In [2]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

target_col = "health_condition"
feature_cols = [c for c in train.columns if c not in [target_col, "id"]]

X = train[feature_cols].copy()
y = train[target_col].copy()
X_test = test[feature_cols].copy()

print("Train shape:", X.shape, " Test shape:", X_test.shape)

Train shape: (690088, 13)  Test shape: (295753, 13)


**Splitting columns into categorical vs numerical.** `select_dtypes` does this automatically based on pandas dtype — text columns (`object`) are categorical, everything else is numeric.

In [3]:
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()

print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)

Categorical columns: ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']
Numeric columns: ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']


**Handling missing values.** Numeric columns get filled with the median (a reasonable, robust default). Categorical columns get filled with the literal string `"missing"` so LightGBM can treat "was missing" as its own category rather than silently dropping the info.

In [4]:
for c in num_cols:
    X[c] = X[c].fillna(X[c].median())
    X_test[c] = X_test[c].fillna(X[c].median())

for c in cat_cols:
    X[c] = X[c].fillna("missing")
    X_test[c] = X_test[c].fillna("missing")

print("Missing values remaining in X:", X.isnull().sum().sum())
print("Missing values remaining in X_test:", X_test.isnull().sum().sum())

Missing values remaining in X: 0
Missing values remaining in X_test: 0


**Setting categorical dtype.** LightGBM needs categorical columns as pandas `category` dtype (not plain strings) to use its native categorical splitting. Train and test must share the exact same category list, or LightGBM will error on any category it hasn't seen before.

In [5]:
for c in cat_cols:
    all_cats = pd.concat([X[c], X_test[c]]).astype(str).unique()
    X[c] = X[c].astype(str).astype(pd.CategoricalDtype(categories=all_cats))
    X_test[c] = X_test[c].astype(str).astype(pd.CategoricalDtype(categories=all_cats))

print("Category dtype set for:", cat_cols)

Category dtype set for: ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


**Label encoding the target.** Models need numbers, not text, so `at-risk` / `unhealthy` / `fit` get mapped to `0` / `1` / `2`.

In [6]:
target_encoder = LabelEncoder()
y_enc = target_encoder.fit_transform(y)

print(dict(zip(target_encoder.classes_, range(len(target_encoder.classes_)))))

# Save the encoded true labels too — useful if I want to validate an ensemble blend later
np.save("y_true_enc.npy", y_enc)

{'at-risk': 0, 'fit': 1, 'unhealthy': 2}


## 2.Tuning with Optuna

Ran 20 trials to search learning rate, num_leaves, depth, min_child_samples, and a couple of regularization terms. Used a single 80/20 train/val split here just to keep the search fast — the real, honest scoring happens after this with proper 5-fold CV.

In [7]:
N_TRIALS = 20

X_search_tr, X_search_val, y_search_tr, y_search_val = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 2000, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 255, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 200, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }
    model = LGBMClassifier(**params)
    model.fit(
        X_search_tr, y_search_tr,
        eval_set=[(X_search_val, y_search_val)],
        callbacks=[early_stopping(stopping_rounds=50, verbose=False), log_evaluation(period=0)]
    )
    val_pred = model.predict(X_search_val)
    return balanced_accuracy_score(y_search_val, val_pred)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print("Best params:", study.best_params)

  0%|          | 0/20 [00:00<?, ?it/s]

Best params: {'n_estimators': 1300, 'learning_rate': 0.01038678600108433, 'num_leaves': 72, 'max_depth': 8, 'min_child_samples': 5, 'subsample': 0.7935047339315643, 'colsample_bytree': 0.6181141323938213, 'reg_alpha': 2.300060214378049e-05, 'reg_lambda': 7.600521043658023e-06}


## 3. Training with the best params (5-fold CV)

Taking whatever Optuna found and running it properly this time — 5-fold stratified CV so the score isn't just one lucky split. Also keeping the out-of-fold predictions around in case I want to compare against other models later.

In [8]:
best_params = study.best_params.copy()
best_params.update({
    "class_weight": "balanced", "random_state": 42, "n_jobs": -1, "verbose": -1,
})

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []
test_preds_folds = []
oof_proba = np.zeros((len(X), 3))   # 3 classes

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_enc)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_enc[train_idx], y_enc[val_idx]

    model = LGBMClassifier(**best_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[early_stopping(stopping_rounds=50, verbose=False), log_evaluation(period=0)]
    )

    val_pred_proba = model.predict_proba(X_val)
    oof_proba[val_idx] = val_pred_proba   # store OOF predictions for these rows

    val_pred = np.argmax(val_pred_proba, axis=1)
    score = balanced_accuracy_score(y_val, val_pred)
    fold_scores.append(score)
    print(f"Fold {fold+1} balanced accuracy: {score:.5f}")

    test_preds_folds.append(model.predict_proba(X_test))

print(f"\nMean CV balanced accuracy: {np.mean(fold_scores):.5f} (+/- {np.std(fold_scores):.5f})")

# Sanity check: OOF-based score should match the fold-average score closely
oof_score = balanced_accuracy_score(y_enc, np.argmax(oof_proba, axis=1))
print(f"OOF balanced accuracy (whole train set): {oof_score:.5f}")

Fold 1 balanced accuracy: 0.95015
Fold 2 balanced accuracy: 0.95169
Fold 3 balanced accuracy: 0.94892
Fold 4 balanced accuracy: 0.94912
Fold 5 balanced accuracy: 0.94764

Mean CV balanced accuracy: 0.94950 (+/- 0.00135)
OOF balanced accuracy (whole train set): 0.94950


## 4. Final predictions

Averaging the 5 fold models' predictions and writing out submission.csv. (Also saving the raw probabilities — handy if I want to revisit ensembling later, but this run stands on its own.)

In [9]:
avg_test_proba = np.mean(test_preds_folds, axis=0)
final_preds_enc = np.argmax(avg_test_proba, axis=1)
final_preds = target_encoder.inverse_transform(final_preds_enc)

submission = pd.DataFrame({"id": test["id"], "health_condition": final_preds})
assert list(submission.columns) == list(sample_sub.columns)
assert len(submission) == len(sample_sub)
submission.to_csv("submission.csv", index=False)

np.save("lightgbm_oof_proba.npy", oof_proba)
np.save("lightgbm_test_proba.npy", avg_test_proba)
print("Saved lightgbm_oof_proba.npy, lightgbm_test_proba.npy, submission.csv")
print("Class order:", target_encoder.classes_)
submission.head()

Saved lightgbm_oof_proba.npy, lightgbm_test_proba.npy, submission.csv
Class order: ['at-risk' 'fit' 'unhealthy']


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
